In [5]:
df

,date,open,high,low,close,volume,tema,trend,tema_diff,prev_trend,trend_change,long_entry,short_entry,atr,label,tema_distance,atr_percent
0,2025-03-01 00:00:00+00:00,2236.59,2236.74,2236.59,2236.59,62.288,2236.590000,None,NaN,None,True,False,False,NaN,0,0.000000,NaN
1,2025-03-01 00:00:05+00:00,2236.59,2237.47,2236.59,2236.62,409.673,2236.593393,UP,0.003393,None,True,True,False,NaN,0,0.000012,NaN
2,2025-03-01 00:00:10+00:00,2236.63,2237.72,2236.47,2237.26,183.504,2236.668903,UP,0.075510,UP,False,False,False,NaN,0,0.000264,NaN
3,2025-03-01 00:00:15+00:00,2237.25,2237.85,2237.25,2237.85,171.908,2236.805292,UP,0.136389,UP,False,False,False,NaN,0,0.000467,NaN
4,2025-03-01 00:00:20+00:00,2237.86,2237.94,2237.79,2237.80,207.517,2236.925396,UP,0.120104,UP,False,False,False,NaN,0,0.000391,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
293755,2025-03-17 23:59:35+00:00,1925.41,1925.42,1925.41,1925.42,5.028,1925.447546,DOWN,-0.044822,DOWN,False,False,False,0.115000,0,-0.000014,0.000060
293756,2025-03-17 23:59:40+00:00,1925.42,1925.42,1925.36,1925.37,15.054,1925.401790,DOWN,-0.045757,DOWN,False,False,False,0.101429,0,-0.000017,0.000053
293757,2025-03-17 23:59:45+00:00,1925.36,1925.37,1925.36,1925.37,33.062,1925.360838,DOWN,-0.040952,DOWN,False,False,False,0.101429,0,0.000005,0.000053
293758,2025-03-17 23:59:50+00:00,1925.36,1925.37,1925.36,1925.37,5.350,1925.324332,DOWN,-0.036506,DOWN,False,False,False,0.096429,0,0.000024,0.000050


In [4]:

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load the data
file_path = "/allah/stuff/freq/project_2/labeled_data/labeled_ETH_USDT_USDT-5s-futures_r1_05.parquet"
df = pd.read_parquet(file_path)
def visualize_entry_sequence(df, entry_idx, window_size=60):
    """Visualize a sequence of candles leading to an entry point"""
    # Get the window of data before the entry
    start_idx = max(0, entry_idx - window_size)
    window_df = df.iloc[start_idx:entry_idx+1].copy()
    
    # Determine entry type
    if df.loc[entry_idx, 'long_entry']:
        entry_type = 'Long'
        color = 'green'
        label_txt = "Profitable" if df.loc[entry_idx, 'label'] == 1 else "Unprofitable"
    else:
        entry_type = 'Short'
        color = 'red'
        label_txt = "Profitable" if df.loc[entry_idx, 'label'] == -1 else "Unprofitable"
    
    # Create the figure
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.05, 
                        subplot_titles=(f'{entry_type} Entry - {label_txt}', 'Volume'),
                        row_heights=[0.8, 0.2])
    
    # Add candlestick chart
    fig.add_trace(go.Candlestick(
        x=window_df.index,
        open=window_df['open'],
        high=window_df['high'],
        low=window_df['low'],
        close=window_df['close'],
        name='Price'
    ), row=1, col=1)
    
    # Add TEMA line
    fig.add_trace(go.Scatter(
        x=window_df.index,
        y=window_df['tema'],
        name='TEMA',
        line=dict(color='purple')
    ), row=1, col=1)
    
    # Highlight the entry point
    fig.add_trace(go.Scatter(
        x=[entry_idx],
        y=[df.loc[entry_idx, 'close']],
        mode='markers',
        marker=dict(color=color, size=12),
        name=f'{entry_type} Entry'
    ), row=1, col=1)
    
    # Add volume bars
    fig.add_trace(go.Bar(
        x=window_df.index,
        y=window_df['volume'],
        name='Volume',
        marker=dict(color='blue')
    ), row=2, col=1)
    
    # Calculate stop loss and take profit levels
    entry_price = df.loc[entry_idx, 'close']
    atr_value = df.loc[entry_idx, 'atr']
    
    if entry_type == 'Long':
        stop_loss = entry_price - (atr_value * 0.5)
        take_profit = entry_price + atr_value
    else:
        stop_loss = entry_price + (atr_value * 0.5)
        take_profit = entry_price - atr_value
    
    # Add horizontal lines for stop loss and take profit
    fig.add_trace(go.Scatter(
        x=[window_df.index[0], window_df.index[-1]],
        y=[stop_loss, stop_loss],
        mode='lines',
        line=dict(color='red', dash='dash'),
        name='Stop Loss'
    ), row=1, col=1)
    
    fig.add_trace(go.Scatter(
        x=[window_df.index[0], window_df.index[-1]],
        y=[take_profit, take_profit],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Take Profit'
    ), row=1, col=1)
    
    # Update layout
    fig.update_layout(
        title=f'Sequence Leading to {entry_type} Entry (Index: {entry_idx})',
        xaxis_rangeslider_visible=False,
        height=800,
        width=1200,
        showlegend=True,
        template='plotly_white'
    )
    
    return fig

# Example usage:
# Find entry points
entry_signals = df[(df['long_entry'] == True) | (df['short_entry'] == True)]

# To visualize a specific entry:
# entry_idx = entry_signals.index[0]  # First entry
# fig = visualize_entry_sequence(df, entry_idx)
# fig.show()
